# 03 Split Folds Make Patches DIBaS

This notebook creates the fixed DIBaS five-fold partition and exports the real `224x224` patches and the cocci-only `256x256` GAN source patches.


## Why This Order Matters

We split raw slides first.

After the fold ids are fixed, we cut patches. This keeps every slide completely inside one fold and prevents slide leakage.


In [1]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import json
import random
import shutil

import pandas as pd

from IPython.display import display
from PIL import Image

# This helper keeps the notebook easy to run from the repo root or from notebooks/.
def find_repo_root(start_path: Path) -> Path:
    if (start_path / "raw_data").exists():
        return start_path
    if start_path.name == "notebooks" and (start_path.parent / "raw_data").exists():
        return start_path.parent
    raise FileNotFoundError("Could not find the FYP2 repo root.")

# This configuration cell locks the DIBaS fold sizes and patch sizes for the whole study.
REPO_ROOT = find_repo_root(Path.cwd())
MANIFESTS_DIR = REPO_ROOT / "manifests"
PATCHES_224_DIR = REPO_ROOT / "patches_224" / "dibas"
GAN_PATCHES_256_DIR = REPO_ROOT / "gan_patches_256" / "dibas"
RESULTS_DIR = REPO_ROOT / "results"
NOTEBOOK_TAG = "03_split_folds_make_patches_dibas"
NOTEBOOK_RESULTS_DIR = RESULTS_DIR / NOTEBOOK_TAG
SEED = 2026
PATCH_SIZE_224 = 224
PATCH_SIZE_256 = 256

MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)
PATCHES_224_DIR.mkdir(parents=True, exist_ok=True)
GAN_PATCHES_256_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# This helper reads a CSV file into a plain list of rows.
def read_csv_rows(csv_path: Path):
    with csv_path.open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

# This helper writes a CSV file with a stable header.
def write_csv_rows(csv_path: Path, rows, fieldnames):
    if not rows:
        raise ValueError(f"No rows were provided for {csv_path.name}.")
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with csv_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

# This helper writes one small JSON file with clean formatting.
def write_json(json_path: Path, data):
    json_path.parent.mkdir(parents=True, exist_ok=True)
    json_path.write_text(json.dumps(data, indent=2), encoding="utf-8")

# This helper stores paths in repo-relative form so the outputs stay portable.
def as_repo_relative(repo_root: Path, path_value: Path) -> str:
    return path_value.relative_to(repo_root).as_posix()

# This helper avoids divide-by-zero problems in small summary tables.
def safe_divide(numerator, denominator):
    return numerator / denominator if denominator else 0.0

# This helper assigns rows to folds by binary label.
# It keeps the split focused on the minority-versus-majority framing of the study.
def assign_rows_to_folds(rows, label_key, sample_key, seed_value, fold_count=5):
    rows_by_label = defaultdict(list)
    for row in rows:
        rows_by_label[row[label_key]].append(dict(row))

    assigned_rows = []

    for label_index, label_value in enumerate(sorted(rows_by_label)):
        label_rows = sorted(rows_by_label[label_value], key=lambda row: row[sample_key])
        random.Random(seed_value + label_index).shuffle(label_rows)
        next_fold_id = 1

        for row in label_rows:
            row_copy = dict(row)
            row_copy["fold_id"] = next_fold_id
            assigned_rows.append(row_copy)
            next_fold_id = (next_fold_id % fold_count) + 1

    if len(assigned_rows) != len(rows):
        raise AssertionError("The fold assignment lost some rows.")

    return assigned_rows

In [3]:
# We load the clean slide manifest from notebook 02 and assign one raw fold id to every slide.
# The split only follows the binary labels.
slide_rows = read_csv_rows(MANIFESTS_DIR / "slide_manifest.csv")
for row in slide_rows:
    row["binary_label"] = int(row["binary_label"])

fold_rows = assign_rows_to_folds(
    rows=slide_rows,
    label_key="binary_label",
    sample_key="slide_id",
    seed_value=SEED,
)

fold_manifest_rows = []
for row in sorted(fold_rows, key=lambda item: (item["fold_id"], item["slide_id"])):
    fold_manifest_rows.append({
        "slide_id": row["slide_id"],
        "species_name": row["species_name"],
        "binary_label": row["binary_label"],
        "binary_group": row["binary_group"],
        "raw_path": row["raw_path"],
        "width": row["width"],
        "height": row["height"],
        "fold_id": row["fold_id"],
    })

fold_manifest_path = MANIFESTS_DIR / "fold_manifest.csv"
write_csv_rows(fold_manifest_path, fold_manifest_rows, list(fold_manifest_rows[0].keys()))

fold_summary_rows = []
for fold_id in range(1, 6):
    rows = [row for row in fold_manifest_rows if int(row["fold_id"]) == fold_id]
    cocci_count = sum(1 for row in rows if int(row["binary_label"]) == 0)
    bacilli_count = sum(1 for row in rows if int(row["binary_label"]) == 1)
    fold_summary_rows.append({
        "fold_id": fold_id,
        "slide_count": len(rows),
        "cocci_slide_count": cocci_count,
        "bacilli_slide_count": bacilli_count,
        "cocci_ratio": round(safe_divide(cocci_count, len(rows)), 6),
        "bacilli_ratio": round(safe_divide(bacilli_count, len(rows)), 6),
    })

species_fold_rows = []
for fold_id in range(1, 6):
    rows = [row for row in fold_manifest_rows if int(row["fold_id"]) == fold_id]
    counts = Counter(row["species_name"] for row in rows)
    for species_name in sorted(counts):
        species_fold_rows.append({
            "fold_id": fold_id,
            "species_name": species_name,
            "slide_count": counts[species_name],
        })

actual_fold_sizes = sorted(row["slide_count"] for row in fold_summary_rows)
if actual_fold_sizes != [29, 29, 29, 29, 29]:
    raise AssertionError(f"Expected DIBaS fold sizes [29, 29, 29, 29, 29], but found {actual_fold_sizes}.")

write_csv_rows(NOTEBOOK_RESULTS_DIR / "fold_summary.csv", fold_summary_rows, list(fold_summary_rows[0].keys()))
write_csv_rows(NOTEBOOK_RESULTS_DIR / "species_by_fold.csv", species_fold_rows, list(species_fold_rows[0].keys()))

display(pd.DataFrame(fold_summary_rows))
display(pd.DataFrame(species_fold_rows))
print(f"Saved fold manifest to: {fold_manifest_path}")

,fold_id,slide_count,cocci_slide_count,bacilli_slide_count,cocci_ratio,bacilli_ratio
0,1,29,12,17,0.413793,0.586207
1,2,29,12,17,0.413793,0.586207
2,3,29,12,17,0.413793,0.586207
3,4,29,12,17,0.413793,0.586207
4,5,29,12,17,0.413793,0.586207


,fold_id,species_name,slide_count
0,1,Clostridium.perfringens,5
1,1,Enterococcus.faecalis,3
2,1,Enterococcus.faecium,4
3,1,Escherichia.coli,3
4,1,Listeria.monocytogenes,7
5,1,Pseudomonas.aeruginosa,2
6,1,Staphylococcus.aureus,5
7,2,Clostridium.perfringens,4
8,2,Enterococcus.faecalis,4
9,2,Enterococcus.faecium,4


Saved fold manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\fold_manifest.csv


In [4]:
# We clear the DIBaS patch folders before writing the new protocol outputs.
if PATCHES_224_DIR.exists():
    shutil.rmtree(PATCHES_224_DIR)
if GAN_PATCHES_256_DIR.exists():
    shutil.rmtree(GAN_PATCHES_256_DIR)
PATCHES_224_DIR.mkdir(parents=True, exist_ok=True)
GAN_PATCHES_256_DIR.mkdir(parents=True, exist_ok=True)

patch_rows_224 = []
gan_patch_rows_256 = []
fold_patch_counts_224 = Counter()
fold_patch_counts_256 = Counter()

for fold_row in fold_manifest_rows:
    fold_id = int(fold_row["fold_id"])
    slide_path = REPO_ROOT / fold_row["raw_path"]
    with Image.open(slide_path) as image_handle:
        image = image_handle.convert("RGB")
        width, height = image.size

        x_steps_224 = width // PATCH_SIZE_224
        y_steps_224 = height // PATCH_SIZE_224
        for y_index in range(y_steps_224):
            for x_index in range(x_steps_224):
                x_start = x_index * PATCH_SIZE_224
                y_start = y_index * PATCH_SIZE_224
                patch = image.crop((x_start, y_start, x_start + PATCH_SIZE_224, y_start + PATCH_SIZE_224))
                patch_id = f"{fold_row['slide_id']}_{PATCH_SIZE_224}_{x_start}_{y_start}"
                target_dir = PATCHES_224_DIR / f"fold_{fold_id}"
                target_dir.mkdir(parents=True, exist_ok=True)
                patch_path = target_dir / f"{patch_id}.png"
                patch.save(patch_path)
                patch_rows_224.append({
                    "dataset_name": "dibas",
                    "patch_id": patch_id,
                    "slide_id": fold_row["slide_id"],
                    "species_name": fold_row["species_name"],
                    "binary_label": fold_row["binary_label"],
                    "binary_group": fold_row["binary_group"],
                    "fold_id": fold_id,
                    "patch_size": PATCH_SIZE_224,
                    "x_start": x_start,
                    "y_start": y_start,
                    "raw_path": fold_row["raw_path"],
                    "file_path": as_repo_relative(REPO_ROOT, patch_path),
                    "source_type": "real",
                })
                fold_patch_counts_224[fold_id] += 1

        if int(fold_row["binary_label"]) == 0:
            x_steps_256 = width // PATCH_SIZE_256
            y_steps_256 = height // PATCH_SIZE_256
            for y_index in range(y_steps_256):
                for x_index in range(x_steps_256):
                    x_start = x_index * PATCH_SIZE_256
                    y_start = y_index * PATCH_SIZE_256
                    patch = image.crop((x_start, y_start, x_start + PATCH_SIZE_256, y_start + PATCH_SIZE_256))
                    patch_id = f"{fold_row['slide_id']}_{PATCH_SIZE_256}_{x_start}_{y_start}"
                    target_dir = GAN_PATCHES_256_DIR / f"fold_{fold_id}"
                    target_dir.mkdir(parents=True, exist_ok=True)
                    patch_path = target_dir / f"{patch_id}.png"
                    patch.save(patch_path)
                    gan_patch_rows_256.append({
                        "dataset_name": "dibas",
                        "gan_patch_id": patch_id,
                        "slide_id": fold_row["slide_id"],
                        "species_name": fold_row["species_name"],
                        "binary_label": fold_row["binary_label"],
                        "binary_group": fold_row["binary_group"],
                        "fold_id": fold_id,
                        "patch_size": PATCH_SIZE_256,
                        "x_start": x_start,
                        "y_start": y_start,
                        "raw_path": fold_row["raw_path"],
                        "file_path": as_repo_relative(REPO_ROOT, patch_path),
                        "source_type": "real_cocci_for_gan",
                    })
                    fold_patch_counts_256[fold_id] += 1

patch_manifest_path = MANIFESTS_DIR / "patch_manifest_224.csv"
gan_patch_manifest_path = MANIFESTS_DIR / "gan_patch_manifest_256.csv"
write_csv_rows(patch_manifest_path, patch_rows_224, list(patch_rows_224[0].keys()))
write_csv_rows(gan_patch_manifest_path, gan_patch_rows_256, list(gan_patch_rows_256[0].keys()))

patch_summary_rows = []
for fold_id in range(1, 6):
    patch_summary_rows.append({
        "fold_id": fold_id,
        "real_patch_count_224": fold_patch_counts_224[fold_id],
        "gan_source_patch_count_256": fold_patch_counts_256[fold_id],
    })
    if fold_patch_counts_224[fold_id] != 1566:
        raise AssertionError(
            f"DIBaS fold {fold_id} should have 1566 real 224 patches, but found {fold_patch_counts_224[fold_id]}."
        )

write_csv_rows(NOTEBOOK_RESULTS_DIR / "patch_summary.csv", patch_summary_rows, list(patch_summary_rows[0].keys()))
write_json(
    NOTEBOOK_RESULTS_DIR / "summary.json",
    {
        "notebook_tag": NOTEBOOK_TAG,
        "seed": SEED,
        "patch_size_224": PATCH_SIZE_224,
        "patch_size_256": PATCH_SIZE_256,
        "fold_manifest_path": as_repo_relative(REPO_ROOT, fold_manifest_path),
        "patch_manifest_path": as_repo_relative(REPO_ROOT, patch_manifest_path),
        "gan_patch_manifest_path": as_repo_relative(REPO_ROOT, gan_patch_manifest_path),
    },
)

display(pd.DataFrame(patch_summary_rows))
print(f"Saved 224 patch manifest to: {patch_manifest_path}")
print(f"Saved 256 GAN patch manifest to: {gan_patch_manifest_path}")

,fold_id,real_patch_count_224,gan_source_patch_count_256
0,1,1566,480
1,2,1566,480
2,3,1566,480
3,4,1566,480
4,5,1566,480


Saved 224 patch manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\patch_manifest_224.csv
Saved 256 GAN patch manifest to: C:\Users\FYP2610\Downloads\FYP2\manifests\gan_patch_manifest_256.csv
